# 39. Reusing prepared inverse-CDF toy generators

**Objectives:**
- Precompute the inverse-transform CDF tables once for a fixed-parameter model with
  `prepare_inverse_toy_generator`.
- Call the returned `PreparedInverseToyGenerator.generate` several times with different
  `n`/`seed` values.
- Compare wall-clock time against calling `generate_toy` fresh every time.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV², daughter
indices start at zero.


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import time

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
    generate_toy, prepare_inverse_toy_generator,
)


## 1. Build a small fixed-parameter model

`docs/toy_generation.md` notes that the expensive part of inverse-transform generation is
*preparation*: evaluating the target density on the tabulation grids. For repeated pseudo-
experiments at the same model parameters, `prepare_inverse_toy_generator` does that work once and
returns a `PreparedInverseToyGenerator` whose `.generate(n, seed=...)` reuses the tables.


In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(0.5, -0.2), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=80, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}
resolution = 256


## 2. Prepare once, generate several times

The model, parameters and resolution must stay fixed while the prepared generator is reused; if
the parameters change, a new `PreparedInverseToyGenerator` must be built (the target CDFs would
otherwise be stale).


In [3]:
prepare_start = time.perf_counter()
prepared = prepare_inverse_toy_generator(
    model, parameters=truth, resolution=resolution,
)
prepare_time = time.perf_counter() - prepare_start
print(f"One-time preparation: {prepare_time * 1000:.1f} ms")

# Warm up JIT compilation once so the timed loop below measures only
# the cost of drawing further toys, not the one-time XLA compile.
_ = prepared.generate(500, seed=0, include_momenta=False)

sizes_and_seeds = [(500, 1), (500, 2), (500, 3), (500, 4), (500, 5)]
prepared_toys = []
prepared_start = time.perf_counter()
for size, seed in sizes_and_seeds:
    prepared_toys.append(prepared.generate(size, seed=seed, include_momenta=False))
prepared_total = time.perf_counter() - prepared_start
print(f"5 reused generate() calls (post warm-up): {prepared_total * 1000:.1f} ms total")
print("Event counts:", [toy.size for toy in prepared_toys])

One-time preparation: 3024.9 ms


5 reused generate() calls (post warm-up): 407.9 ms total
Event counts: [500, 500, 500, 500, 500]


## 3. Compare against calling generate_toy fresh each time

`generate_toy(..., method="inverse-transform")` repeats the full preparation internally on every
call. With modest settings the absolute times are small, but the reused generator should still be
noticeably faster in total than repeating preparation for every pseudoexperiment.


In [4]:
fresh_start = time.perf_counter()
fresh_toys = []
for size, seed in sizes_and_seeds:
    fresh_toys.append(
        generate_toy(
            model, size, parameters=truth, seed=seed,
            method="inverse-transform", inverse_resolution=resolution,
            include_momenta=False,
        )
    )
fresh_total = time.perf_counter() - fresh_start
print(f"5 fresh generate_toy() calls: {fresh_total * 1000:.1f} ms total")
print(f"Reused-generator total (prepare once + 5 generate calls): {(prepare_time + prepared_total) * 1000:.1f} ms")
speedup = fresh_total / prepared_total if prepared_total > 0 else float("nan")
print(f"Warmed-up generate() calls alone are ~{speedup:.1f}x faster in total than repeating full preparation each time")
assert prepared_total < fresh_total, "reusing the prepared generator should be cheaper for repeated draws"

5 fresh generate_toy() calls: 736.6 ms total
Reused-generator total (prepare once + 5 generate calls): 3432.8 ms
Warmed-up generate() calls alone are ~1.8x faster in total than repeating full preparation each time


## Summary and exercises

1. `prepare_inverse_toy_generator` tabulates the signal (and optional background) CDFs once;
   `PreparedInverseToyGenerator.generate` reuses them for arbitrary `n`/`seed` combinations.
2. Rebuild the prepared generator whenever the model's floating parameters change — a stale
   generator silently samples from the wrong density.
3. Increase `resolution` and re-run: preparation grows more expensive but each `generate` call
   still amortizes it.
4. `benchmarks/benchmark_toy_generation.py` reports the same preparation-vs-reuse comparison at
   production scale.

Reference: [toy generation](../../docs/toy_generation.md), "Reuse the prepared inverse CDFs".

Return to [the course guide](TUTORIALS.md).
